# Прогнозирование конечных свойств композиционных материалов

**Автор:** Жихарев Вячеслав Сергеевич

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
matplotlib.style.use('seaborn-pastel')
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# загрузка данных из локальной папки с новым датасетом
def read_composite_part(path):
    data = pd.read_excel(path)
    if 'Unnamed: 0' in data.columns:
        return data.set_index('Unnamed: 0')

    unnamed_cols = [col for col in data.columns if str(col).startswith('Unnamed')]
    if unnamed_cols:
        data = data.drop(columns=unnamed_cols)

    return data

bp = read_composite_part('hw_data_composite/X_bp.xlsx')
nup = read_composite_part('hw_data_composite/X_nup.xlsx')

# объединение по индексу, тип INNER
df = bp.join(nup, how='inner').reset_index(drop=True)

# если после первого ноутбука есть очищенный датасет, используем его
clean_dataset_path = 'Dataset_composites_inner_clean.csv'
if os.path.exists(clean_dataset_path):
    df = pd.read_csv(clean_dataset_path)
    print(f'Используется очищенный датасет: {clean_dataset_path}, shape={df.shape}')
else:
    print(f'Очищенный датасет не найден, используется полный датасет, shape={df.shape}')

In [ ]:
df

# Эксперименты с исходными данными

In [ ]:
# #   
# df['Содержание матрицы, %'] = df['Соотношение матрица-наполнитель'] / ((df['Соотношение матрица-наполнитель'] + 1) / 100)
# df['Содержание наполнителя, %'] = 100 - df['Содержание матрицы, %']
# # Эксперименты с использованием этих колонок не дали улучшения результатов

In [ ]:
# # У некоторых колонок непонятные единицы измерения (в частности: "%_2")
# # Предполагая, что это значения в квадрате, изменим их.
# df['Содержание эпоксидных групп,%_2'] = df['Содержание эпоксидных групп,%_2'] ** 0.5
# df['Содержание эпоксидных групп,%_2'].describe().T.round(2)
# # Эксперименты с использованием этих колонок не дали улучшения результатов

Эксперименты с модификацией исходных данных не дали улучшения качества.

Для дальнейшего моделирования используем исходный набор признаков.

# Построение моделей

In [ ]:
#Нормализация данных
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df_norm = MinMaxScaler().fit_transform(np.array(df))
df_norm = pd.DataFrame(data = df_norm, columns = df.columns)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, SGDRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

## Модели для таргета `Модуль упругости при растяжении, ГПа`

In [ ]:
# Задаем Х у
X = df_norm[['Соотношение матрица-наполнитель', 'Количество отвердителя, м.%', 
             'Содержание эпоксидных групп,%_2', 'Потребление смолы, г/м2', 'Угол нашивки, град', 'Шаг нашивки', 'Плотность нашивки']]
y = df_norm['Модуль упругости при растяжении, ГПа']

# X = df_norm[['Соотношение матрица-наполнитель', 'Плотность, кг/м3', 'модуль упругости, ГПа', 'Количество отвердителя, м.%', 
#              'Содержание эпоксидных групп,%_2', 'Температура вспышки, С_2', 'Поверхностная плотность, г/м2', 'Потребление смолы, г/м2', 
#              'Угол нашивки, град', 'Шаг нашивки', 'Плотность нашивки']]
# y = df_norm['Модуль упругости при растяжении, ГПа']

# X = df[['Соотношение матрица-наполнитель', 'Количество отвердителя, м.%', 
#              'Содержание эпоксидных групп,%_2', 'Потребление смолы, г/м2', 'Угол нашивки, град', 'Шаг нашивки', 'Плотность нашивки']]
# y = df['Модуль упругости при растяжении, ГПа']

# X = df[['Соотношение матрица-наполнитель', 'Плотность, кг/м3', 'модуль упругости, ГПа', 'Количество отвердителя, м.%', 
#              'Содержание эпоксидных групп,%_2', 'Температура вспышки, С_2', 'Поверхностная плотность, г/м2', 'Потребление смолы, г/м2', 
#              'Угол нашивки, град', 'Шаг нашивки', 'Плотность нашивки']]
# y = df['Модуль упругости при растяжении, ГПа']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

### Набор моделей регрессии

In [ ]:
models = [
    {'model': LinearRegression(), 
     'params': {'fit_intercept': [True, False], 'copy_X': [True, False]}},
    {'model': Ridge(), 
     'params': {'alpha': [0.1, 1, 10, 100],}},
    {'model': Lasso(), 
     'params': {'alpha': [0.1, 1, 10, 100],}},
    {'model': ElasticNet(), 
     'params': {'alpha': [0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]}},
    {'model': SVR(),
     'params': {'kernel': ['linear', 'rbf', 'sigmoid'],'C': [0.1, 1, 10, 100],'epsilon': [0.1, 0.2, 0.3],'gamma': ['scale', 'auto']}},
    {'model': RandomForestRegressor(), 
     'params': {'n_estimators': [10, 20, 30], 'max_depth': [3, 5, 7], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}},
    {'model': AdaBoostRegressor(), 
     'params': {'n_estimators': [10, 20, 30], 'learning_rate': [0.1, 1, 10], 'loss' : ['linear', 'square', 'exponential']}},
    {'model': GradientBoostingRegressor(), 
     'params': {'learning_rate': [0.1, 1, 10],'n_estimators': [10, 20, 30],'subsample' : [0.5, 0.8, 1.0],'max_depth': [3, 5, 7],'min_samples_split':[2, 5, 10],'min_samples_leaf':[1, 2, 4],'random_state': [42]}},
    {'model': DecisionTreeRegressor(),
     'params': {'max_depth' : range(1,50),'min_samples_split' : range(2,50),'min_samples_leaf' : range(1,50)}},
    {'model': XGBRegressor(),
     'params': {'learning_rate': [0.1, 1, 10],'n_estimators': [10, 20, 30],'subsample' : [0.5, 0.8, 1.0],'max_depth': [3, 5, 7],'min_child_weight':[1, 2, 4],'gamma':[0, 0.1, 0.5, 1],'random_state': [42]}},
    {'model': SGDRegressor(),
     'params': {'penalty': ['l1', 'l2', 'elasticnet'],'alpha': [0.0001, 0.001],'epsilon': [0.1, 0.2, 0.3],'learning_rate': ['constant', 'optimal', 'invscaling'], 'shuffle': [True, False], 'loss':['squared_loss', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive']}},
    {'model': BayesianRidge(),
     'params': {'n_iter': [100, 200, 300], 'tol': [0.001, 0.0001, 0.00001]}},
    {'model': KNeighborsRegressor(),
     'params': {'n_neighbors': [1, 3, 5], 'weights': ['uniform', 'distance'], 'algorithm': ['auto', 'ball_tree', 'kd_tree'], 'leaf_size': [20, 30, 40]}},
]

for model in models:
    print(model['model'].__class__.__name__)
    classifier = GridSearchCV(model['model'], model['params'], cv=10)
    classifier.fit(X_train, y_train)

    y_pred_train = classifier.predict(X_train)
    y_pred_test = classifier.predict(X_test)

    print('Best parameters:', classifier.best_params_)
    print(f"Train MSE: {mean_squared_error(y_train, y_pred_train):.3f}")
    print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.3f}")
    print(f"Train MAE: {mean_absolute_error(y_train, y_pred_train):.3f}")
    print(f"Train R2: {r2_score(y_train, y_pred_train):.3f}")

    print(f"Test MSE: {mean_squared_error(y_test, y_pred_test):.3f}")
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.3f}")
    print(f"Test MAE: {mean_absolute_error(y_test, y_pred_test):.3f}")
    print(f"Test R2: {r2_score(y_test, y_pred_test):.3f}")
    print('===')

## Модели для таргета `Прочность при растяжении, МПа`

In [ ]:
# Задаем X и y для прогноза прочности при растяжении
X = df[['Соотношение матрица-наполнитель', 'Плотность, кг/м3', 'модуль упругости, ГПа', 'Количество отвердителя, м.%', 
        'Содержание эпоксидных групп,%_2', 'Температура вспышки, С_2', 'Поверхностная плотность, г/м2', 'Потребление смолы, г/м2', 
        'Угол нашивки, град', 'Шаг нашивки', 'Плотность нашивки']]
y = df['Прочность при растяжении, МПа']

# по ТЗ: 30% данных оставляем для теста
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
models = [
    {'model': LinearRegression(), 
     'params': {'fit_intercept': [True, False], 'copy_X': [True, False]}},
    {'model': Ridge(), 
     'params': {'alpha': [0.1, 1, 10, 100],}},
    {'model': Lasso(), 
     'params': {'alpha': [0.1, 1, 10, 100],}},
    {'model': ElasticNet(), 
     'params': {'alpha': [0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]}},
    {'model': SVR(),
     'params': {'kernel': ['linear', 'rbf', 'sigmoid'],'C': [0.1, 1, 10, 100],'epsilon': [0.1, 0.2, 0.3],'gamma': ['scale', 'auto']}},
    {'model': RandomForestRegressor(), 
     'params': {'n_estimators': [10, 20, 30], 'max_depth': [3, 5, 7], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}},
    {'model': AdaBoostRegressor(), 
     'params': {'n_estimators': [10, 20, 30], 'learning_rate': [0.1, 1, 10], 'loss' : ['linear', 'square', 'exponential']}},
    {'model': GradientBoostingRegressor(), 
     'params': {'learning_rate': [0.1, 1, 10],'n_estimators': [10, 20, 30],'subsample' : [0.5, 0.8, 1.0],'max_depth': [3, 5, 7],'min_samples_split':[2, 5, 10],'min_samples_leaf':[1, 2, 4],'random_state': [42]}},
    {'model': DecisionTreeRegressor(),
     'params': {'max_depth' : range(1,50),'min_samples_split' : [2, 5, 10, 20, 30, 40, 50],'min_samples_leaf' : [1, 5, 10, 20, 30, 40, 50]}},
    {'model': XGBRegressor(),
     'params': {'learning_rate': [0.1, 1, 10],'n_estimators': [10, 20, 30],'subsample' : [0.5, 0.8, 1.0],'max_depth': [3, 5, 7],'min_child_weight':[1, 2, 4],'gamma':[0, 0.1, 0.5, 1],'random_state': [42]}},
    {'model': SGDRegressor(),
     'params': {'penalty': ['l1', 'l2', 'elasticnet'],'alpha': [0.0001, 0.001],'epsilon': [0.1, 0.2, 0.3],'learning_rate': ['constant', 'optimal', 'invscaling'], 'shuffle': [True, False], 'loss':['squared_loss', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive']}},
    {'model': BayesianRidge(),
     'params': {'n_iter': [100, 200, 300], 'tol': [0.001, 0.0001, 0.00001]}},
    {'model': KNeighborsRegressor(),
     'params': {'n_neighbors': [1, 3, 5], 'weights': ['uniform', 'distance'], 'algorithm': ['auto', 'ball_tree', 'kd_tree'], 'leaf_size': [20, 30, 40]}},
]
for model in models:
    print(model['model'].__class__.__name__)
    classifier = GridSearchCV(model['model'], model['params'], cv=10)
    classifier.fit(X_train, y_train)

    y_pred_train = classifier.predict(X_train)
    y_pred_test = classifier.predict(X_test)

    print('Best parameters:', classifier.best_params_)
    print(f"Train MSE: {mean_squared_error(y_train, y_pred_train):.3f}")
    print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.3f}")
    print(f"Train MAE: {mean_absolute_error(y_train, y_pred_train):.3f}")
    print(f"Train R2: {r2_score(y_train, y_pred_train):.3f}")

    print(f"Test MSE: {mean_squared_error(y_test, y_pred_test):.3f}")
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.3f}")
    print(f"Test MAE: {mean_absolute_error(y_test, y_pred_test):.3f}")
    print(f"Test R2: {r2_score(y_test, y_pred_test):.3f}")
    print('===')

In [ ]:
plt.scatter(y_test, y_pred)

Построены 13 моделей регрессии для каждого таргета (линейная регрессия, Ridge, Lasso, ElasticNet, SVR, случайный лес, бустинги и др.).

Лучшие результаты показали ансамблевые методы и градиентный бустинг.

In [ ]:
# сохраняем лучшую модель локально для CLI-приложения
import pickle
with open('best_regressor.pkl', 'wb') as f:
    pickle.dump(classifier, f)